# 13 Canopy Ensemble

Ensemble from baseline + random forest + boosting notebooks.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import importlib

# Resolve project root in local or Kaggle runtime.
if Path("/kaggle/working").exists():
    ROOT = Path("/kaggle/working")
else:
    ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()

if (ROOT / "src").exists():
    SRC = ROOT / "src"
else:
    SRC = Path.cwd() / "src"
if str(SRC) not in sys.path and SRC.exists():
    sys.path.append(str(SRC))

if importlib.util.find_spec("laspy") is None:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "laspy[lazrs]"])

from lidar_roraima.config import ProjectConfig
cfg = ProjectConfig.from_root(ROOT)
cfg.ensure_dirs()

# Kaggle dataset fallback path.
RAW_DATA_DIR = cfg.raw_data_dir
for candidate in [
    Path("/kaggle/input/lidar-roraima-parime-research/lidar_data"),
    Path("/kaggle/input/lidar-roraima-parime-research"),
]:
    if candidate.exists():
        RAW_DATA_DIR = candidate
        break

print("ROOT:", ROOT)
print("RAW_DATA_DIR:", RAW_DATA_DIR)
TRAIN_MAX_ROWS = None  # Set integer for constrained runtime, e.g. 1200000
cfg


In [ ]:
import pandas as pd
from lidar_roraima.features import load_feature_tables
from lidar_roraima.inference import run_inference
from lidar_roraima.ensemble import blend_regression_predictions

features = load_feature_tables(cfg.features_dir)
model_paths = [
    cfg.models_dir / "canopy_baseline.joblib",
    cfg.models_dir / "canopy_random_forest.joblib",
    cfg.models_dir / "canopy_boosting.joblib",
]
preds = [run_inference(path, features, prediction_column="pred_chm", uncertainty_column="uncertainty_chm_single") for path in model_paths if path.exists()]
ensemble = blend_regression_predictions(preds, pred_col="pred_chm")
ensemble.to_parquet(cfg.inference_dir / "canopy_ensemble.parquet", index=False)
ensemble.head()
